### Clasificación de Noticias - Deep Learning

In [2]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
# Cargamos los datos (asumiendo el mismo CSV de la práctica anterior)
df = pd.read_csv('df_total.csv', sep=",")

# Convertimos las etiquetas de texto (ej: "deportes") a números (0, 1, 2...)
le = LabelEncoder()
df['label_num'] = le.fit_transform(df['Type'])
num_clases = len(le.classes_)

#### Tokenización y Padding

In [3]:
# Parámetros de configuración
vocab_size = 10000 # Solo usamos las 10,000 palabras más comunes
max_length = 200 # Cortamos o rellenamos las noticias a 200 palabras

trunc_type = 'post'
padding_type = 'post'
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(df['news'])

# Convertimos texto a secuencias de números
sequences = tokenizer.texts_to_sequences(df['news'])

# Aseguramos que todas midan lo mismo
padded = pad_sequences(sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)


#### División de datos

In [4]:
X_train, X_test, y_train, y_test = train_test_split(padded, df['label_num'], test_size=0.2, random_state=42)

#### Creación de la Red Neuronal

In [5]:
model = tf.keras.Sequential([
# 1. Capa de Embedding: Crea un espacio vectorial para las palabras
tf.keras.layers.Embedding(vocab_size, 16, input_length=max_length),

# 2. Capa GlobalAveragePooling: Reduce la dimensionalidad
tf.keras.layers.GlobalAveragePooling1D(),

# 3. Capa Densa: Una capa intermedia de neuronas para aprender patrones
tf.keras.layers.Dense(24, activation='relu'),

# 4. Capa de Salida: Una neurona por cada categoría (deportes, política, etc.)
 tf.keras.layers.Dense(num_clases, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()


/home/ciabd10/anaconda3/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
W0000 00:00:1774364902.250285    4462 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

#### Entrenamiento

In [8]:
history = model.fit(X_train, y_train, epochs=40, validation_data=(X_test, y_test), verbose=2)

Epoch 1/40


31/31 - 0s - 4ms/step - accuracy: 0.5920 - loss: 1.2141 - val_accuracy: 0.5574 - val_loss: 1.2977
Epoch 2/40
31/31 - 0s - 3ms/step - accuracy: 0.6053 - loss: 1.1514 - val_accuracy: 0.5820 - val_loss: 1.2463
Epoch 3/40
31/31 - 0s - 4ms/step - accuracy: 0.6249 - loss: 1.0843 - val_accuracy: 0.5697 - val_loss: 1.2065
Epoch 4/40
31/31 - 0s - 3ms/step - accuracy: 0.6629 - loss: 1.0206 - val_accuracy: 0.5943 - val_loss: 1.1619
Epoch 5/40
31/31 - 0s - 3ms/step - accuracy: 0.6680 - loss: 0.9496 - val_accuracy: 0.6393 - val_loss: 1.0988
Epoch 6/40
31/31 - 0s - 3ms/step - accuracy: 0.7061 - loss: 0.8728 - val_accuracy: 0.6434 - val_loss: 1.0508
Epoch 7/40
31/31 - 0s - 4ms/step - accuracy: 0.7369 - loss: 0.8064 - val_accuracy: 0.6557 - val_loss: 0.9937
Epoch 8/40
31/31 - 0s - 3ms/step - accuracy: 0.8068 - loss: 0.7380 - val_accuracy: 0.6721 - val_loss: 0.9488
Epoch 9/40
31/31 - 0s - 3ms/step - accuracy: 0.8243 - loss: 0.6747 - val_accuracy: 0.7049 - val_loss: 0.9100
Epoch 10/40
31/31 - 0s - 4ms/s

#### Evaluación y Predicción

In [9]:
# Evaluamos la precisión final
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Precisión del modelo Deep Learning: {accuracy*100:.2f}%")

# Ejemplo de predicción con una noticia nueva
nueva_noticia = ["El equipo ganó el campeonato de liga en el último minuto"]

secuencia_nueva = tokenizer.texts_to_sequences(nueva_noticia)
padded_nueva = pad_sequences(secuencia_nueva, maxlen=max_length)

prediccion = model.predict(padded_nueva)
clase_predicha = le.inverse_transform([tf.argmax(prediccion[0]).numpy()])

print(f"La noticia se clasifica como: {clase_predicha[0]}")

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8238 - loss: 0.4940 
Precisión del modelo Deep Learning: 82.38%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
La noticia se clasifica como: Macroeconomia
